# YOLOv3 + Tucker Compression — Unified Demo Notebook

End-to-end in one notebook: train a YOLOv3 detector, compress it with the
Tucker pipeline (same methodology as our CNN/ViT work), and analyze the result.

**Structure (watch the tags):**
- **Part 1–2** — Setup, data, model (fast)
- **Part 3 — Training** ⏱️ SLOW — pre-computed, skip live
- **Part 4 — Phase 1 sweep** ⏱️ SLOW — pre-computed, skip live
- **Part 5 — Phase 2 selection** ⚡ LIVE — run this in the demo
- **Part 6 — Analysis + architecture visuals** — the payoff

**Demo safety:** the `SKIP_SLOW_STAGES` flag below makes the slow cells
(training, Phase 1) no-op so a stray "Run All" won't stall the demo. They
load from your saved checkpoints instead. Set it `False` only when you
actually want to (re)train or (re)sweep.

## Master setup — run once, everything below uses these

In [ ]:
import sys, os, json, glob, time
sys.path.append(os.path.abspath("../src"))
import numpy as np
import torch
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# demo-safety flag: True = skip training + Phase 1 (load from checkpoints)
SKIP_SLOW_STAGES = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| SKIP_SLOW_STAGES:", SKIP_SLOW_STAGES)

In [ ]:
# All pipeline function imports (consolidated) — run once near the top
from model import YOLOv3, count_conv_params
from dataset import CocoSubsetDataset, yolo_collate_fn
from tucker_pipeline import (
    get_compressible_layers, fit_biquadratic_polynomial,
    select_ranks_for_layer, apply_selected_ranks,
    recalibrate_batchnorm, get_module_by_name, tucker_decompose_layer,
)
from tucker_phase1 import evaluate_map, run_phase1
from postprocess import predict_boxes
from analysis_utils import count_params, model_size_mb, measure_latency
from train_utils import fit

---
# Part 1 — Data Loading

In [ ]:
from dataset import CocoSubsetDataset, yolo_collate_fn

## Config — edit these for your machine / demo

In [ ]:
DATA_ROOT = "../data/coco"          # expects DATA_ROOT/annotations, DATA_ROOT/train2017, DATA_ROOT/val2017

# Automotive / driving-relevant COCO classes (8): road users + traffic infrastructure
CLASS_NAMES = ["person", "bicycle", "car", "motorcycle", "bus", "truck", "traffic light", "stop sign"]

IMAGES_PER_CLASS = 2000             # cap per class; common classes (car/person) hit it,
                                    # rarer ones (stop sign) take whatever COCO has. ~16k images.
IMG_SIZE = 416
BATCH_SIZE = 8

In [ ]:
train_ds = CocoSubsetDataset(DATA_ROOT, "train2017", CLASS_NAMES,
                              img_size=IMG_SIZE, images_per_class=IMAGES_PER_CLASS, augment=True)
val_ds = CocoSubsetDataset(DATA_ROOT, "val2017", CLASS_NAMES,
                            img_size=IMG_SIZE, images_per_class=max(20, IMAGES_PER_CLASS // 5), augment=False)

print(f"train images: {len(train_ds)}  val images: {len(val_ds)}  classes: {train_ds.num_classes}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           collate_fn=yolo_collate_fn, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=yolo_collate_fn, num_workers=0)

## Sanity check — visualize one batch with its boxes

In [ ]:
imgs, targets = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i >= imgs.shape[0]:
        ax.axis("off"); continue
    img = imgs[i].permute(1, 2, 0).numpy()
    ax.imshow(img)
    for box in targets[i]:
        cls, cx, cy, w, h = box.tolist()
        x0, y0 = (cx - w/2) * IMG_SIZE, (cy - h/2) * IMG_SIZE
        rect = plt.Rectangle((x0, y0), w * IMG_SIZE, h * IMG_SIZE, fill=False, edgecolor="red", linewidth=2)
        ax.add_patch(rect)
        ax.text(x0, y0 - 4, CLASS_NAMES[int(cls)], color="red", fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Save the config for the other notebooks

So `02_model_definition`, `03_training`, `04_decomposition`, `05_analysis` all agree on
`CLASS_NAMES` / `IMG_SIZE` without re-typing them.

In [ ]:
import json
os.makedirs("../checkpoints", exist_ok=True)
with open("../checkpoints/run_config.json", "w") as f:
    json.dump({"class_names": CLASS_NAMES, "img_size": IMG_SIZE,
               "data_root": DATA_ROOT, "batch_size": BATCH_SIZE}, f, indent=2)
print("saved ../checkpoints/run_config.json")

---
# Part 2 — Model Definition

In [ ]:
from model import YOLOv3, count_conv_params
NUM_CLASSES = len(CLASS_NAMES)
print("classes:", CLASS_NAMES)

In [ ]:
model = YOLOv3(num_classes=NUM_CLASSES)
total, conv = count_conv_params(model)
print(f"total params: {total:,}")
print(f"conv params:  {conv:,}  ({conv/total:.1%} of total)")
print(f"num Conv2d layers: {sum(1 for m in model.modules() if isinstance(m, torch.nn.Conv2d))}")

## Forward-pass shape check (3 output scales: stride 32 / 16 / 8)

In [ ]:
x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
with torch.no_grad():
    outs = model(x)
for scale_name, o in zip(["large (stride32)", "medium (stride16)", "small (stride8)"], outs):
    print(f"{scale_name}: {tuple(o.shape)}   -> anchors*(5+{NUM_CLASSES}) = {3*(5+NUM_CLASSES)} channels")

## Save an untrained checkpoint (useful as a reset point for the demo)

In [ ]:
os.makedirs("../checkpoints", exist_ok=True)
torch.save(model.state_dict(), "../checkpoints/yolov3_untrained.pt")
print("saved ../checkpoints/yolov3_untrained.pt")

---
# Part 3 — Training  ⏱️ SLOW (pre-computed)

Guarded by `SKIP_SLOW_STAGES`. With the flag `True`, these cells don't
retrain — the model is loaded from your saved checkpoint in later parts.
Set the flag `False` (top of notebook) only to actually train.

## Training config — the knobs you'd actually tune before the demo run

In [ ]:
EPOCHS = 120
LEARNING_RATE = 1e-3   # higher base LR; the warmup+cosine schedule handles it
CKPT_DIR = "../checkpoints"
IMAGES_PER_CLASS = 2000   # match notebook 01

train_ds = CocoSubsetDataset(DATA_ROOT, "train2017", CLASS_NAMES, img_size=IMG_SIZE,
                              images_per_class=IMAGES_PER_CLASS, augment=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                           collate_fn=yolo_collate_fn, num_workers=0)

model = YOLOv3(num_classes=NUM_CLASSES)

# --- Pretrained backbone (ImageNet Darknet-53 via timm) ---
# Initializes model.backbone from pretrained weights; detection neck/heads
# stay random and train from scratch. This is the single biggest lever for
# reaching good mAP. Set USE_PRETRAINED_BACKBONE=False to train fully from
# scratch (much harder to converge to high mAP).
USE_PRETRAINED_BACKBONE = True
if USE_PRETRAINED_BACKBONE:
    from pretrained_backbone import load_pretrained_backbone
    load_pretrained_backbone(model)

model = model.to(DEVICE)

In [ ]:
if not SKIP_SLOW_STAGES:
    history = fit(model, train_loader, DEVICE, epochs=EPOCHS, lr=LEARNING_RATE,
                  ckpt_dir=CKPT_DIR, num_classes=NUM_CLASSES, resume=True,
                  warmup_epochs=3, use_schedule=True)
else:
    print("SKIP_SLOW_STAGES=True -> skipping training; using saved checkpoint")

## Loss curve — sanity check that training actually converged before the demo

In [ ]:
losses = [h["loss"] for h in history]
plt.figure(figsize=(8, 4))
plt.plot(losses, marker="o")
plt.xlabel("epoch"); plt.ylabel("avg training loss"); plt.title("YOLOv3 training loss")
plt.grid(alpha=0.3)
plt.show()
print(f"final loss: {losses[-1]:.4f}  (best: {min(losses):.4f})")

## Output

`../checkpoints/yolov3_best.pt` — the trained weights the decomposition
notebook loads. `../checkpoints/history.json` — full loss history if you
want to show the convergence plot live without re-running training.

---
# Part 4 — Phase 1 Sweep  ⏱️ SLOW (pre-computed)

Guarded. `run_phase1` already resumes from saved per-layer JSON checkpoints,
so even if run it loads instantly. With `SKIP_SLOW_STAGES=True` it's skipped
entirely and Part 5 reads the stored `checkpoints/phase1/` results.

## Load the trained baseline model

In [ ]:
model = YOLOv3(num_classes=NUM_CLASSES)
state = torch.load("../checkpoints/yolov3_best.pt", map_location=DEVICE)
model.load_state_dict(state["model_state"])
model.to(DEVICE).eval()
print("loaded trained baseline")

## Build the validation loader + stratified sample

The **val loader** feeds the (subsampled) mAP evaluation. The **stratified
sample** feeds the noise metric — a handful of images per class is plenty,
since each image contributes hundreds of thousands of activation values to
the std-based noise estimate (same rationale as the CNN pipeline's 1-per-class).

In [ ]:
val_ds = CocoSubsetDataset(DATA_ROOT, "val2017", CLASS_NAMES, img_size=IMG_SIZE,
                            images_per_class=30, augment=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=yolo_collate_fn, num_workers=0)

# stratified sample for noise: reuse a few val images per class
strat_ds = CocoSubsetDataset(DATA_ROOT, "val2017", CLASS_NAMES, img_size=IMG_SIZE,
                              images_per_class=2, augment=False)
X_sample = torch.stack([strat_ds[i][0] for i in range(len(strat_ds))], dim=0)
print("stratified sample:", tuple(X_sample.shape))

## Baseline mAP (uncompressed reference)

Measured once up front — Phase 2 uses this as the reference the accuracy budget is subtracted from.

In [ ]:
baseline_map = evaluate_map(model, val_loader, NUM_CLASSES, DEVICE, IMG_SIZE, max_batches=None)
print(f"baseline mAP@0.5: {baseline_map:.4f}")

os.makedirs("../checkpoints/phase1", exist_ok=True)
with open("../checkpoints/phase1/baseline.json", "w") as f:
    json.dump({"baseline_map": baseline_map}, f)

## Enumerate compressible layers (37 for YOLOv3: 3×3 convs, skip stem + prediction heads)

In [ ]:
compressible = get_compressible_layers(model, skip_first=True)
print(f"{len(compressible)} compressible layers")
total = 0
for l in compressible:
    step = max(1, int(round(0.20 * min(l["C_in"], l["C_out"]))))
    import math
    n_in = len(list(range(step, l["C_in"]+1, step))) + (1 if l["C_in"] % step else 0)
    n_out = len(list(range(step, l["C_out"]+1, step))) + (1 if l["C_out"] % step else 0)
    total += n_in * n_out
print(f"~{total} total (r_in, r_out) combinations across all layers")

## Run Phase 1

⚠️ **This is the long-running cell.** Faithful to the CNN pipeline, it
deep-copies the full model per combination and evaluates mAP + noise. On the
DGX GPU this is feasible offline but not fast — leave it running. It
checkpoints after every layer to `../checkpoints/phase1/phase1_layerNNN.json`
and **resumes automatically** if interrupted (skips layers already done).

Tune `MAP_MAX_BATCHES` down to speed up the sweep (fewer val batches per mAP
estimate) or up for more accurate curves. Since Phase 1 is pre-run offline,
you can afford a reasonably large value here for trustworthy polynomials.

In [ ]:
if not SKIP_SLOW_STAGES:
    MAP_MAX_BATCHES = 20
    NOISE_BATCH_SIZE = 8
    layer_results = run_phase1(
        model, compressible, X_sample, val_loader, NUM_CLASSES, DEVICE, IMG_SIZE,
        ckpt_dir="../checkpoints/phase1",
        map_max_batches=MAP_MAX_BATCHES, noise_batch_size=NOISE_BATCH_SIZE)
    print(f"Phase 1 done: {len(layer_results)} layers")
else:
    print("SKIP_SLOW_STAGES=True -> skipping Phase 1 sweep; Part 5 loads saved results")

## Quick look: noise-vs-mAP curve for one layer

Sanity-check that the fitted biquadratic tracks the swept points.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

li = next(l for l in layer_results if l["poly_coeffs"] is not None)
sweep = [r for r in li["sweep_results"] if not np.isnan(r["noise_percent"])]
noise = [r["noise_percent"] for r in sweep]
mAP = [r["mAP"] for r in sweep]

xs = np.linspace(0, max(noise), 200)
poly = np.poly1d(li["poly_coeffs"])
plt.figure(figsize=(8, 5))
plt.scatter(noise, mAP, alpha=0.6, label="swept combos")
plt.plot(xs, poly(xs), "r-", label="biquadratic fit")
plt.xlabel("noise %"); plt.ylabel("mAP@0.5"); plt.title(f"Layer: {li['name']}")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

## Output

`../checkpoints/phase1/phase1_layerNNN.json` (one per layer) + `baseline.json`.
These are the only things Phase 2 needs — carry this folder to the demo machine.

---
# Part 5 — Phase 2 Selection  ⚡ LIVE

This is the fast, interactive part — pure math on the stored Phase 1 results.
Change `BUDGET_PCT` / `TIER` and re-run to show different compression/accuracy
tradeoffs live.

## Load stored Phase 1 results

In [ ]:
layer_results = []
for path in sorted(glob.glob("../checkpoints/phase1/phase1_layer*.json")):
    with open(path) as f:
        layer_results.append(json.load(f))
with open("../checkpoints/phase1/baseline.json") as f:
    baseline_map = json.load(f)["baseline_map"]

print(f"loaded {len(layer_results)} layers, baseline mAP@0.5 = {baseline_map:.4f}")

## Set the global budget and run selection

`BUDGET_PCT` is a **percentage of baseline mAP** you're willing to give up
(e.g. 5.0 = allow the fitted curve to drop to 95% of baseline mAP). Change
this live to show tighter vs looser compression.

In [ ]:
BUDGET_PCT = 5.0
quality_budget = baseline_map * (BUDGET_PCT / 100.0)   # absolute mAP drop allowed
print(f"budget: {BUDGET_PCT}% of baseline -> allow up to {quality_budget:.4f} mAP drop")

selections_all = []
for lr in layer_results:
    if lr["poly_coeffs"] is None:
        selections_all.append({"layer": lr["name"], "max_noise": None, "suggestions": []})
        continue
    # baseline reference on this layer's own curve = poly value at noise=0
    layer_baseline = float(np.poly1d(lr["poly_coeffs"])(0))
    sel = select_ranks_for_layer(lr["name"], lr["sweep_results"], lr["poly_coeffs"],
                                  layer_baseline, quality_budget)
    selections_all.append(sel)

n_with = sum(1 for s in selections_all if s["suggestions"])
print(f"{n_with}/{len(selections_all)} layers have suggestions within budget")

## Inspect suggestions for a few layers

In [ ]:
for sel in selections_all[:5]:
    if not sel["suggestions"]:
        print(f"{sel['layer']}: (no suggestion within budget)")
        continue
    print(f"\n{sel['layer']}  (max tolerable noise {sel['max_noise']:.1f}%)")
    for s in sel["suggestions"]:
        print(f"  {s['label']:13s} r_in={s['r_in']:4d} r_out={s['r_out']:4d} "
              f"noise={s['noise_percent']:5.1f}% mAP={s['mAP']:.4f} CR={s['compression_ratio']:.2f}x")

## Pick a tier and build the per-layer rank map

Choose one tier globally (`Conservative` / `Balanced` / `Aggressive`). For
each layer we take its suggestion at that tier (falling back to the closest
available if that layer's front has fewer than 3 clusters).

In [ ]:
TIER = "Balanced"

selected_ranks = {}
for sel in selections_all:
    if not sel["suggestions"]:
        continue
    match = [s for s in sel["suggestions"] if s["label"] == TIER]
    chosen = match[0] if match else sel["suggestions"][len(sel["suggestions"]) // 2]
    selected_ranks[sel["layer"]] = {"r_in": chosen["r_in"], "r_out": chosen["r_out"]}

print(f"selected ranks for {len(selected_ranks)} layers at tier '{TIER}'")

## Apply decomposition to the model

In [ ]:
model = YOLOv3(num_classes=NUM_CLASSES)
state = torch.load("../checkpoints/yolov3_best.pt", map_location=DEVICE)
model.load_state_dict(state["model_state"])
model.to(DEVICE).eval()

from analysis_utils import count_params
p_before = count_params(model)
model = apply_selected_ranks(model, selected_ranks, DEVICE)
p_after = count_params(model)
print(f"params: {p_before:,} -> {p_after:,}  ({p_after/p_before:.1%} kept)")

## Recalibrate BatchNorm

Essential after decomposition — resyncs BN running stats to the factored
layers' outputs (forward-only, no weight changes). Without this the
compressed model produces erratic, image-dependent detection counts.

In [ ]:
recal_ds = CocoSubsetDataset(DATA_ROOT, "train2017", CLASS_NAMES, img_size=IMG_SIZE,
                              images_per_class=30, augment=False)
recal_loader = DataLoader(recal_ds, batch_size=BATCH_SIZE, shuffle=True,
                           collate_fn=yolo_collate_fn, num_workers=0)
model = recalibrate_batchnorm(model, recal_loader, DEVICE, num_batches=20)
print("BatchNorm recalibrated")

## Optional fine-tuning of the compressed model  ⏱️ SLOW (pre-run)

Fine-tunes the decomposed model at a low LR so weights adapt to compression,
recovering accuracy beyond BN recalibration alone. Guarded by
`SKIP_SLOW_STAGES` — pre-run offline, load the result for the demo. Shows mAP
before vs after so the recovery is visible. `FINETUNE_EPOCHS = 0` skips it.

In [ ]:
from tucker_pipeline import finetune_compressed

FINETUNE_EPOCHS = 10               # 0 = skip; pre-run offline when > 0
FINETUNE_IMAGES_PER_CLASS = 2000   # full for recovery; lower for speed
FINETUNE_LR = 1e-4

_val_ds_ft = CocoSubsetDataset(DATA_ROOT, "val2017", CLASS_NAMES, img_size=IMG_SIZE,
                               images_per_class=30, augment=False)
_val_loader_ft = DataLoader(_val_ds_ft, batch_size=BATCH_SIZE, shuffle=False,
                            collate_fn=yolo_collate_fn, num_workers=0)
map_before_ft = evaluate_map(model, _val_loader_ft, NUM_CLASSES, DEVICE, IMG_SIZE, max_batches=None)
print(f"mAP before fine-tuning: {map_before_ft:.4f}")

In [ ]:
if FINETUNE_EPOCHS > 0 and not SKIP_SLOW_STAGES:
    _ft_ds = CocoSubsetDataset(DATA_ROOT, "train2017", CLASS_NAMES, img_size=IMG_SIZE,
                               images_per_class=FINETUNE_IMAGES_PER_CLASS, augment=True)
    _ft_loader = DataLoader(_ft_ds, batch_size=BATCH_SIZE, shuffle=True,
                            collate_fn=yolo_collate_fn, num_workers=0)
    model = finetune_compressed(model, _ft_loader, DEVICE, epochs=FINETUNE_EPOCHS,
                                 num_classes=NUM_CLASSES, lr=FINETUNE_LR,
                                 ckpt_dir="../checkpoints/finetune", resume=True)
    map_after_ft = evaluate_map(model, _val_loader_ft, NUM_CLASSES, DEVICE, IMG_SIZE, max_batches=None)
    print(f"mAP after fine-tuning:  {map_after_ft:.4f}  ({map_after_ft-map_before_ft:+.4f})")
elif FINETUNE_EPOCHS > 0 and SKIP_SLOW_STAGES:
    # load pre-fine-tuned compressed model if it was pre-run
    import os
    ft_ckpt = "../checkpoints/finetune/yolov3_best.pt"
    if os.path.exists(ft_ckpt):
        model.load_state_dict(torch.load(ft_ckpt, map_location=DEVICE)["model_state"])
        model.eval()
        print("loaded pre-fine-tuned compressed model from checkpoint")
    else:
        print("SKIP_SLOW_STAGES=True but no fine-tune checkpoint found; using recalibrated model")
else:
    print("FINETUNE_EPOCHS=0 -> fine-tuning skipped")

## Save the compressed model

In [ ]:
torch.save(model.state_dict(), "../checkpoints/yolov3_compressed.pt")
with open("../checkpoints/compression_selection.json", "w") as f:
    json.dump({"tier": TIER, "budget_pct": BUDGET_PCT, "selected_ranks": selected_ranks}, f, indent=2)
print("saved ../checkpoints/yolov3_compressed.pt")

x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)
with torch.no_grad():
    outs = model(x)
print("output shapes:", [tuple(o.shape) for o in outs])

---
# Part 6 — Analysis

## Load both models

In [ ]:
state = torch.load("../checkpoints/yolov3_best.pt", map_location="cpu")
original = YOLOv3(num_classes=NUM_CLASSES)
original.load_state_dict(state["model_state"])
original.to(DEVICE).eval()

compressed = YOLOv3(num_classes=NUM_CLASSES)
compressed.load_state_dict(state["model_state"])
compressed = apply_selected_ranks(compressed, selected_ranks, DEVICE)
compressed.load_state_dict(torch.load("../checkpoints/yolov3_compressed.pt", map_location="cpu"))
compressed.to(DEVICE).eval()
print("both models loaded")

## Params, size, latency

In [ ]:
p_o, p_c = count_params(original), count_params(compressed)
s_o, s_c = model_size_mb(original), model_size_mb(compressed)
l_o = measure_latency(original, (1,3,IMG_SIZE,IMG_SIZE), DEVICE)
l_c = measure_latency(compressed, (1,3,IMG_SIZE,IMG_SIZE), DEVICE)

print(f"{'':16s}{'original':>14s}{'compressed':>14s}{'change':>12s}")
print(f"{'params':16s}{p_o:>14,d}{p_c:>14,d}{(1-p_c/p_o)*100:>11.1f}%")
print(f"{'size (MB)':16s}{s_o:>14.1f}{s_c:>14.1f}{(1-s_c/s_o)*100:>11.1f}%")
print(f"{'latency (ms)':16s}{l_o:>14.2f}{l_c:>14.2f}{(1-l_c/l_o)*100:>+11.1f}%")

Latency note: Tucker replaces one conv with three sequential convs, so wall-clock speed doesn't always improve in proportion to parameter reduction — the win here is model size / memory footprint. State this explicitly if asked.

## mAP@0.5 comparison

In [ ]:
val_ds = CocoSubsetDataset(DATA_ROOT, "val2017", CLASS_NAMES, img_size=IMG_SIZE,
                            images_per_class=30, augment=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=yolo_collate_fn, num_workers=0)

map_o = evaluate_map(original, val_loader, NUM_CLASSES, DEVICE, IMG_SIZE, max_batches=None)
map_c = evaluate_map(compressed, val_loader, NUM_CLASSES, DEVICE, IMG_SIZE, max_batches=None)
print(f"original mAP@0.5:   {map_o:.4f}")
print(f"compressed mAP@0.5: {map_c:.4f}  ({map_c-map_o:+.4f})")

## Side-by-side detections on one image

In [ ]:
def draw(ax, img_t, boxes, scores, labels, title):
    ax.imshow(img_t.permute(1,2,0).cpu().numpy())
    for b, s, l in zip(boxes, scores, labels):
        x1,y1,x2,y2 = b.tolist()
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,fill=False,edgecolor="lime",linewidth=2))
        ax.text(x1, max(y1-4,0), f"{CLASS_NAMES[int(l)]} {s:.2f}", color="black",
                fontsize=8, bbox=dict(facecolor="lime", alpha=0.7, pad=1))
    ax.set_title(f"{title} ({len(boxes)} det)"); ax.axis("off")

img_t, _ = val_ds[0]
batch = img_t.unsqueeze(0).to(DEVICE)
with torch.no_grad():
    po = predict_boxes(original, batch, NUM_CLASSES, conf_thresh=0.5)[0]
    pc = predict_boxes(compressed, batch, NUM_CLASSES, conf_thresh=0.5)[0]

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
draw(axes[0], img_t, *po, "Original")
draw(axes[1], img_t, *pc, "Compressed")
plt.tight_layout(); plt.show()

## Summary table

In [ ]:
import pandas as pd
pd.DataFrame({
    "metric": ["Total params", "Model size (MB)", "Latency (ms/img)", "mAP@0.5"],
    "original": [f"{p_o:,}", f"{s_o:.1f}", f"{l_o:.2f}", f"{map_o:.4f}"],
    "compressed": [f"{p_c:,}", f"{s_c:.1f}", f"{l_c:.2f}", f"{map_c:.4f}"],
    "change": [f"{(1-p_c/p_o)*100:.1f}%", f"{(1-s_c/s_o)*100:.1f}%",
               f"{(1-l_c/l_o)*100:+.1f}%", f"{map_c-map_o:+.4f}"],
})

---
# Part 7 — Architecture Visualization (before / after decomposition)

Three ways to see what Tucker decomposition does to the architecture:
1. **torchview** — inline graphs, no server (best for live demo)
2. **ONNX + Netron inline** — interactive, needs a forwarded port on the DGX
3. **ONNX + Netron static image** — zero-risk fallback (pre-export the PNGs)

Both **whole-model** (the big picture) and a **single layer** (the clear
conv → 1×1/k×k/1×1 TuckerBlock transformation) are shown.

In [ ]:
from viz_utils import (torchview_graph, single_layer_before_after,
                       export_onnx, netron_inline, netron_image)
from tucker_pipeline import get_module_by_name, tucker_decompose_layer

## 7a. Single layer — the clearest view of what decomposition does

One 3×3 conv becomes three convs (1×1 reduce → 3×3 core → 1×1 expand). This is the whole idea in one picture.

In [ ]:
# pick one compressed layer and build its original-vs-TuckerBlock pair
demo_layer = list(selected_ranks.keys())[0]
ranks = selected_ranks[demo_layer]
orig_conv = get_module_by_name(original, demo_layer)
block = tucker_decompose_layer(orig_conv, ranks["r_in"], ranks["r_out"])
one_conv, one_block = single_layer_before_after(original, demo_layer, block)

print(f"layer: {demo_layer}")
print(f"  original: 1 conv  {orig_conv.in_channels}->{orig_conv.out_channels}, 3x3")
print(f"  tucker:   1x1({orig_conv.in_channels}->{ranks['r_in']}) -> "
      f"3x3({ranks['r_in']}->{ranks['r_out']}) -> 1x1({ranks['r_out']}->{orig_conv.out_channels})")

In [ ]:
# BEFORE: single conv
in_ch = orig_conv.in_channels
torchview_graph(one_conv, input_size=(1, in_ch, 32, 32), depth=2,
                graph_name="original_layer").visual_graph

In [ ]:
# AFTER: the 3-conv TuckerBlock
torchview_graph(one_block, input_size=(1, in_ch, 32, 32), depth=3,
                graph_name="tucker_layer").visual_graph

## 7b. Whole model — the big picture (original vs compressed)

Larger graphs; `depth` controls detail. Lower depth = higher-level overview.

In [ ]:
# original whole model (keep depth modest so the graph stays readable)
torchview_graph(original, input_size=(1,3,IMG_SIZE,IMG_SIZE), depth=1,
                device=DEVICE, graph_name="yolov3_original").visual_graph

In [ ]:
# compressed whole model
torchview_graph(compressed, input_size=(1,3,IMG_SIZE,IMG_SIZE), depth=1,
                device=DEVICE, graph_name="yolov3_compressed").visual_graph

## 7c. ONNX export (for real Netron)

Writes ONNX files you can open in Netron — the desktop app, or the browser
at netron.app (drag the file in). Needed for both the inline-iframe and
static-image Netron options below.

In [ ]:
os.makedirs("../checkpoints/onnx", exist_ok=True)
export_onnx(original,   "../checkpoints/onnx/yolov3_original.onnx",   device="cpu")
export_onnx(compressed, "../checkpoints/onnx/yolov3_compressed.onnx", device="cpu")

## 7d. Netron inline (interactive) — optional

Embeds the interactive Netron viewer in the cell. Needs port 8080 reachable;
on the DGX over VS Code Remote-SSH the port must be forwarded (VS Code often
does it automatically). If the frame is blank, the port isn't forwarded — use
the static image (7e) instead.

In [ ]:
# netron_inline("../checkpoints/onnx/yolov3_original.onnx", port=8080)
# (uncomment to try; leaves a server running on that port)

## 7e. Netron static image — zero-risk demo fallback

Pre-export a PNG from Netron once (open the ONNX in the Netron app →
Export → PNG, or the download button on netron.app), save it to
`../checkpoints/onnx/`, then display it inline. Nothing runs live, so it
always shows — the safest choice for the demo.

In [ ]:
# after exporting from Netron, e.g.:
# netron_image("../checkpoints/onnx/yolov3_original_netron.png")